# Example of using the `Streetscape` class

The class is designed for morphological streetscape analysis, focusing on generating nd analyzing streetscape measures based on sight points and sightlines. It attempts to capture the space from the pedestrian point of view.

For details and citation, refer to *Araldi, A., Fleischmann, M., Fusco, G., Novotný, M., 2025. Streetscape morphometrics: Expanding momepy to analyze urban form from the street point of view. SoftwareX 31, 102242. [https://doi.org/10.1016/j.softx.2025.102242](https://doi.org/10.1016/j.softx.2025.102242).*

In [ ]:
import geopandas as gpd
import momepy
import numpy as np
import osmnx as ox
import rioxarray

Read all the data. Only streets and buildings are required.

In [ ]:
streets = gpd.read_file(
    momepy.datasets.get_path("bubenec"), layer="streets"
).to_crs(5514)
buildings = gpd.read_file(
    momepy.datasets.get_path("bubenec"), layer="buildings"
).to_crs(5514)
plots = gpd.read_file(
    momepy.datasets.get_path("bubenec"), layer="plots"
).to_crs(5514)
dtm = rioxarray.open_rasterio(momepy.datasets.get_path("bubenec"), layer="dtm")

Mimic data on building category and height.

In [ ]:
buildings["category"] = np.random.randint(0, 6, len(buildings))
buildings["height"] = np.random.randint(12, 30, len(buildings))

Initiate the class. This will dricectly compute builk of the sightline indicators based on streets and buildings.

In [ ]:
sc = momepy.Streetscape(
    streets, buildings, category_col="category", height_col="height"
)

If you have plots and DTM, you can use two additional methods to compute additional variables.

In [ ]:
sc.compute_plots(plots)
sc.compute_slope(dtm)

The resulting data can be extracted either on a street level:

In [ ]:
street_df = sc.street_level()
street_df.head()

It is a GeoDataFrame, so you can directly plot it.

In [ ]:
ax = street_df.plot("os", figsize=(12, 12), legend=True)
buildings.plot(ax=ax).set_axis_off()

Or for all individual sightline points.

In [ ]:
point_df = sc.point_level()
point_df.head(10)

Again, it is a GeoDataFrame, this time with point geometry.

In [ ]:
ax = point_df.plot("os", figsize=(12, 12), legend=True, markersize=5)
buildings.plot(ax=ax).set_axis_off()

## Using OpenStreetMap data

You can also use any other data. If you fetch buildings and streets from OSM, you can measure a subset of characters depending on those.

In [ ]:
gdf = ox.features_from_place("Kahla, Germany", tags={"building": True})
buildings = ox.projection.project_gdf(gdf).reset_index()

streets_graph = ox.graph_from_place("Kahla, Germany", network_type="drive")
streets_graph = ox.projection.project_graph(streets_graph)
edges = ox.graph_to_gdfs(
    streets_graph,
    nodes=False,
    edges=True,
    node_geometry=False,
    fill_edge_geometry=True,
).reset_index(drop=True)

Initiate the class. This time without additional arguments as OSM does not contain category or reliable height.

In [ ]:
sc = momepy.Streetscape(edges, buildings)

The measurememnt happens on class initialisation. Now you can simply extract it either on a street level:

In [ ]:
street_df = sc.street_level()

ax = street_df.plot("os", figsize=(12, 12), legend=True)
buildings.plot(ax=ax, color="lightgray").set_axis_off()

Or on a point level, as before.

In [ ]:
point_df = sc.point_level()

ax = point_df.plot("os", figsize=(12, 12), legend=True, markersize=0.1)
buildings.plot(ax=ax, color="lightgray").set_axis_off()